# 02. Subsets and Combinations

Selecting **some** items (order does not matter). Core ideas: the `start` index, sorting to handle duplicates, and pruning.

**Learning order:** DUPLICATES → COMBINATIONS → COMBINATION SUM → COMBINATION SUM II → PHONE LETTERS → INTERVIEW THINKING

## 1. Subsets II (input has duplicates)

**Sort** first so equal values are neighbours. At the same recursion level, **skip** a value if it equals the previous one (`i > start and nums[i] == nums[i-1]`). The first copy is allowed to extend into deeper levels (`[2, 2]`), but we never start a new branch with the same value twice on the same level.

In [1]:
def subsets_with_dup(nums):
    nums.sort()
    res, path = [], []
    def backtrack(start):
        res.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:      # skip duplicate at the same level
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()
    backtrack(0)
    return res

print(subsets_with_dup([1, 2, 2]))   # [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]

[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]


## 2. Combinations: choose k from 1..n

Same as subsets but save only when `len(path) == k`. **Pruning:** if the numbers left are not enough to reach size `k`, stop: `i` can go only up to `n - (k - len(path)) + 1`.

In [2]:
def combine(n, k):
    res, path = [], []
    def backtrack(start):
        if len(path) == k:
            res.append(path[:])
            return
        for i in range(start, n - (k - len(path)) + 2):   # prune: need enough numbers left
            path.append(i)
            backtrack(i + 1)
            path.pop()
    backtrack(1)
    return res

print(combine(4, 2))   # [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]

[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]


## 3. Combination Sum (reuse allowed)

Pick numbers (each **unlimited times**) that sum to `target`. Recurse with `backtrack(i, remaining)`, passing `i` (not `i + 1`) because the same number can be used again. **Sort** and `break` when a candidate exceeds `remaining` (all later ones are bigger).

In [3]:
def combination_sum(candidates, target):
    candidates.sort()
    res, path = [], []
    def backtrack(start, remaining):
        if remaining == 0:
            res.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:              # prune
                break
            path.append(candidates[i])
            backtrack(i, remaining - candidates[i])    # i, not i+1: reuse allowed
            path.pop()
    backtrack(0, target)
    return res

print(combination_sum([2, 3, 6, 7], 7))   # [[2, 2, 3], [7]]

[[2, 2, 3], [7]]


## 4. Combination Sum II (each number once, duplicates in input)

Sort, use `i + 1` (no reuse), and skip duplicates at the same level.

In [4]:
def combination_sum2(candidates, target):
    candidates.sort()
    res, path = [], []
    def backtrack(start, remaining):
        if remaining == 0:
            res.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remaining:
                break
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            path.append(candidates[i])
            backtrack(i + 1, remaining - candidates[i])
            path.pop()
    backtrack(0, target)
    return res

print(combination_sum2([10, 1, 2, 7, 6, 1, 5], 8))   # [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]

[[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]


## 5. Letter Combinations of a Phone Number

Each digit maps to letters. At each position **choose one letter** from the digit's letters. Depth = number of digits. Real life: all possible words typed on an old keypad.

In [5]:
def letter_combinations(digits):
    if not digits:
        return []
    phone = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl",
             "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
    res = []
    def backtrack(i, cur):
        if i == len(digits):
            res.append(cur)
            return
        for ch in phone[digits[i]]:
            backtrack(i + 1, cur + ch)       # strings are immutable: no undo needed
    backtrack(0, "")
    return res

print(letter_combinations("23"))   # ['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']

['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


## 6. Interview thinking

**Cheat rules**
- Order does not matter → use `start` and recurse with `i + 1` (or `i` if reuse is allowed).
- Input has duplicates → **sort** and `if i > start and nums[i] == nums[i-1]: continue`.
- Sums/targets → sort and **break** early when a number is too big.

**Common mistakes:** using `used` array in combinations (that is for permutations), duplicate check using `i > 0` instead of `i > start`, forgetting to sort, using `i` vs `i + 1` incorrectly.

## Quick Revision

- Subsets: record at every node. Combinations: record at size k.
- `start` index = no reordering. `i+1` = use once, `i` = reuse.
- Duplicates: sort + skip `i > start and same as previous`.
- Prune: not enough elements left, or candidate > remaining.